# Notebook 11: TRÍCH XUẤT SIÊU TỐC ẢNH PATCHES & FILE SVS CHO 12 CA BỆNH VÀNG
---
## 🎯 Mục Đích của Notebook này trên Kaggle
Trích xuất nhanh trực tiếp thư mục ảnh vi thể thật và file `.svs` của **12 Ca Bệnh Chuẩn Vàng (100% Chính Xác)** để nạp vào giao diện Web Demo CDSS:

### 🏆 12 Ca Bệnh Chuẩn Vàng (Golden Patients):
- **3 Ca Luminal A:** `TCGA-OL-A66J`, `TCGA-AR-A5QM`, `TCGA-E2-A15P`
- **3 Ca Luminal B:** `TCGA-A8-A08P`, `TCGA-BH-A18L`, `TCGA-A8-A092`
- **3 Ca Basal-like:** `TCGA-AQ-A04J`, `TCGA-AN-A0FX`, `TCGA-A2-A04T`
- **3 Ca HER2-enriched:** `TCGA-C8-A12Z`, `TCGA-C8-A12Q`, `TCGA-C8-A275`

In [ ]:
# =========================================================================
# 1. TRÍCH XUẤT SIÊU TỐC TRONG 3 GIÂY
# =========================================================================
import os
import shutil
import time

start_time = time.time()

GOLDEN_PATIENTS = {
    'LumA': ['TCGA-OL-A66J', 'TCGA-AR-A5QM', 'TCGA-E2-A15P'],
    'LumB': ['TCGA-A8-A08P', 'TCGA-BH-A18L', 'TCGA-A8-A092'],
    'Basal': ['TCGA-AQ-A04J', 'TCGA-AN-A0FX', 'TCGA-A2-A04T'],
    'HER2': ['TCGA-C8-A12Z', 'TCGA-C8-A12Q', 'TCGA-C8-A275']
}
all_golden_pids = [pid for p_list in GOLDEN_PATIENTS.values() for pid in p_list]

STAGE_DIR = '/kaggle/working/wsi_visual_assets'
if os.path.exists(STAGE_DIR):
    shutil.rmtree(STAGE_DIR)
os.makedirs(f"{STAGE_DIR}/data/wsi_patches", exist_ok=True)
os.makedirs(f"{STAGE_DIR}/data/raw_svs", exist_ok=True)

print("⏳ Đang định vị trực tiếp thư mục Input...")
input_datasets = [os.path.join('/kaggle/input', d) for d in os.listdir('/kaggle/input')]
print(f"✓ Các Dataset Input: {[os.path.basename(d) for d in input_datasets]}")

# A. TRÍCH XUẤT ẢNH PATCHES TẾ BÀO (Lấy 30 ảnh cho mỗi ca vàng)
copied_patients = set()
for inp in input_datasets:
    for root, dirs, files in os.walk(inp):
        matched_dirs = [d for d in dirs if any(pid in d for pid in all_golden_pids)]
        for d in matched_dirs:
            pid = [p for p in all_golden_pids if p in d][0]
            if pid in copied_patients: continue
            src_folder = os.path.join(root, d)
            dst_folder = f"{STAGE_DIR}/data/wsi_patches/{pid}"
            os.makedirs(dst_folder, exist_ok=True)
            img_list = [f for f in os.listdir(src_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:30]
            for idx, im in enumerate(img_list):
                shutil.copy(os.path.join(src_folder, im), os.path.join(dst_folder, f"{pid}_patch_{idx:03d}.png"))
            print(f"  • [Patch] Đã sao chép {len(img_list):>2} ảnh mô học của bệnh nhân: {pid}")
            copied_patients.add(pid)
        if len(copied_patients) >= len(all_golden_pids):
            break

# B. TRÍCH XUẤT 1 FILE SVS MẪU
for inp in input_datasets:
    for root, dirs, files in os.walk(inp):
        svs_list = [f for f in files if f.lower().endswith('.svs')]
        if svs_list:
            target_svs = svs_list[0]
            full_svs_path = os.path.join(root, target_svs)
            sz_mb = os.path.getsize(full_svs_path) / (1024 * 1024)
            print(f"\n✓ Tìm thấy {len(svs_list)} file SVS. File mẫu: {target_svs} ({sz_mb:.1f} MB)")
            if sz_mb < 200:
                shutil.copy(full_svs_path, f"{STAGE_DIR}/data/raw_svs/{target_svs}")
                print(f"  🔬 Đã sao chép {target_svs} vào gói zip!")
            else:
                print(f"  ℹ️ File SVS {target_svs} dung lượng lớn ({sz_mb:.1f} MB), có thể tải trực tiếp từ giao diện Kaggle khi cần.")
            break

# C. NÉN GÓI ZIP 1-CLICK
ZIP_OUT = '/kaggle/working/wsi_visual_assets.zip'
print(f"\n⏳ Đang nén file ZIP: {ZIP_OUT}...")
shutil.make_archive('/kaggle/working/wsi_visual_assets', 'zip', STAGE_DIR)

elapsed = time.time() - start_time
zip_size_mb = os.path.getsize(ZIP_OUT) / (1024 * 1024)

print(f"=========================================================================")
print(f"🎉 HOÀN THÀNH XUẤT SẮC TRONG: {elapsed:.2f} GIÂY!")
print(f"📦 File Zip: {ZIP_OUT} ({zip_size_mb:.2f} MB)")
print(f"=========================================================================")
print("👉 Bạn vào Tab Output bên phải, tải file wsi_visual_assets.zip về máy!")